# AgroPest-12 — Crop Pest Detection with YOLOv8s

End-to-end walkthrough: download the dataset, fine-tune YOLOv8s, evaluate detection
and classification performance, and inspect what the model learned.

Designed to run on a free Google Colab **T4** GPU. Runtime → Change runtime type → T4 GPU.

The heavy lifting lives in `src/agropest/` so the same code runs from the command line;
this notebook is the narrated version.

## 1. Environment

In [ ]:
!pip install -q ultralytics scikit-learn kagglehub pyyaml

import torch
print("torch:", torch.__version__)
print("CUDA :", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")

In [ ]:
# Make the project package importable.
# On Colab: !git clone https://github.com/<your-username>/agropest-12-yolov8.git
import sys
from pathlib import Path

REPO = Path("..").resolve()      # adjust if you cloned elsewhere
sys.path.insert(0, str(REPO / "src"))

from agropest.config import TrainConfig, EvalConfig
from agropest.data import load_class_names, write_data_yaml, summarise_split

## 2. Dataset

AgroPest-12 already ships in YOLO format, so preparation is just pointing a
`data.yaml` at the extracted archive. Kaggle credentials are required — upload
`kaggle.json`, or set `KAGGLE_USERNAME` / `KAGGLE_KEY`.

**Never commit `kaggle.json`.** It is in `.gitignore` for this reason.

In [ ]:
import kagglehub

DATASET_ROOT = Path(kagglehub.dataset_download("rupankarmajumdar/crop-pests-dataset"))
print("Extracted to:", DATASET_ROOT)
print(sorted(p.name for p in DATASET_ROOT.iterdir()))

In [ ]:
# Class names come from the dataset, never hard-coded — see README
# "A note on class labels" for what happens when they are.
CLASS_NAMES = load_class_names(REPO / "data" / "data.yaml")
DATA_YAML = write_data_yaml(DATASET_ROOT, CLASS_NAMES, REPO / "data" / "data.yaml")

print(DATA_YAML.read_text())

In [ ]:
for split in ("train", "valid", "test"):
    info = summarise_split(DATASET_ROOT, split)
    print(f"{info['split']:<6} {info['n_images']:>6} images  "
          f"{sum(info['instances_per_class'].values()):>6} instances")

### Class balance

Mild imbalance, not severe enough to warrant resampling.

In [ ]:
import matplotlib.pyplot as plt

counts = summarise_split(DATASET_ROOT, "train")["instances_per_class"]
ordered = sorted(counts.items(), key=lambda kv: -kv[1])

fig, ax = plt.subplots(figsize=(10, 4))
ax.bar([CLASS_NAMES[i] for i, _ in ordered], [c for _, c in ordered], color="#2f6f4e")
ax.set_ylabel("Instances")
ax.set_title("Training set class distribution")
ax.spines[["top", "right"]].set_visible(False)
plt.xticks(rotation=45, ha="right")
plt.tight_layout()

## 3. Training

50 epochs at 512x512 with mosaic, horizontal flip and HSV jitter. Early stopping
(patience 15) on validation mAP — in the reported run it triggered near epoch 30.

512x512 is a hardware compromise for the T4, not an optimum. Insects in this
dataset are small, and YOLO downsamples early, so a larger input would very
likely improve recall. See the README's Limitations section.

Expect ~2 hours on a T4.

In [ ]:
from agropest.train import train

cfg = TrainConfig(epochs=50, imgsz=512, batch=16, patience=15,
                  device="0" if torch.cuda.is_available() else "cpu")
model, minutes, run_dir = train(DATA_YAML, cfg)

BEST = run_dir / "weights" / "best.pt"
print(f"{minutes:.1f} min -> {BEST}")

## 4. Evaluation

Three families of numbers:

1. **Detection** — mAP@0.5, mAP@0.5:0.95, precision, recall.
2. **Classification** — image-level top-1, so the detector can be compared
   like-for-like against classical baselines that only emit a label.
3. **Efficiency** — latency and FPS.

In [ ]:
from agropest.evaluate import evaluate

results = evaluate(BEST, DATA_YAML, EvalConfig(split="test"))

In [ ]:
import pandas as pd

per_class = pd.DataFrame(results["detection"]["per_class"]).T
per_class.sort_values("AP@0.5").style.background_gradient(cmap="Greens", subset=["AP@0.5"])

In [ ]:
report = pd.DataFrame(results["classification"]["report"]).T
report.loc[CLASS_NAMES].round(4)

### Reading the results

Detection PR-AUC sits well below classification PR-AUC because detection scores
localisation *and* labelling together. Once the model finds an insect it names it
correctly ~92% of the time; putting a tight box around a 20-pixel thrip against
cluttered foliage is the harder half of the problem.

## 5. Figures

In [ ]:
from agropest.visualize import plot_confusion_matrix, plot_per_class_ap, plot_predictions
from ultralytics import YOLO

FIG = REPO / "results" / "figures"
TEST_DIR = DATASET_ROOT / "test"

plot_confusion_matrix(results["classification"]["confusion_matrix"], CLASS_NAMES,
                      FIG / "confusion_matrix.png")
plot_per_class_ap(results["detection"]["per_class"], FIG / "per_class_ap.png")
plot_predictions(YOLO(str(BEST)), TEST_DIR, FIG / "predictions.png", n=9)

In [ ]:
from IPython.display import Image, display

for name in ("confusion_matrix.png", "per_class_ap.png", "predictions.png"):
    display(Image(filename=str(FIG / name), width=700))

## 6. Interpretability — Grad-CAM

Grad-CAM confirms the model attends to the same cues a human would: ant leg
joints and head/thorax junctions, caterpillar dorsal hairs, grasshopper antennae
and hind legs, snail shell spirals.

It also explains the weak classes. Beetles are distinguished by leg groupings and
short antennae, weevils by dorsal texture — so front and rear views of the two
collapse together. Earthworms and slugs confuse for the mirror-image reason:
both are segmented, limbless and textureless.

In [ ]:
!pip install -q grad-cam
# Target the backbone conv blocks and overlay CAMs on test images.
# Target layers: convolutional blocks 6, 8, 12, 18 and the third-from-last.

## 7. Export

`best.pt` is the artefact worth keeping. Model weights are gitignored — publish
them as a GitHub Release or on the Hugging Face Hub rather than committing them.

In [ ]:
import json, shutil

RESULTS = REPO / "results"
(RESULTS / "metrics.json").write_text(json.dumps(results, indent=2))
shutil.copy(BEST, RESULTS / "best.pt")
print("Saved metrics.json and best.pt")